# Codificación aritmética

Este cuaderno implementa un codificador aritmético desde cero y lo compara
con la codificación de Huffman en términos de eficiencia.

**Artículo relacionado:** `02/13-codificacion-aritmetica.md`

In [ ]:
import math
from collections import Counter
import heapq
import numpy as np

## 1. Codificador aritmético (precisión real)

Implementamos el codificador con números de punto flotante para claridad conceptual.
La implementación entera se muestra en la sección 3.

In [ ]:
def arithmetic_encode(message, probs):
    """
    Codifica un mensaje usando codificación aritmética.
    
    Args:
        message: lista de símbolos a codificar
        probs: dict {símbolo: probabilidad}
    
    Returns:
        (low, high): intervalo final
        bits_needed: número de bits necesarios para representar el código
    """
    # Construir tabla de intervalos acumulada
    symbols = sorted(probs.keys())
    cum_probs = {}
    cum = 0.0
    for s in symbols:
        cum_probs[s] = (cum, cum + probs[s])
        cum += probs[s]
    
    low, high = 0.0, 1.0
    steps = []
    
    for symbol in message:
        width = high - low
        s_low, s_high = cum_probs[symbol]
        new_low = low + width * s_low
        new_high = low + width * s_high
        steps.append({
            'symbol': symbol,
            'interval_before': (low, high),
            'interval_after': (new_low, new_high)
        })
        low, high = new_low, new_high
    
    # Longitud del intervalo = probabilidad del mensaje
    prob_message = high - low
    bits_needed = math.ceil(-math.log2(prob_message)) + 1
    
    return low, high, bits_needed, steps


def arithmetic_decode(code_value, probs, n_symbols):
    """
    Decodifica un valor real a una secuencia de n_symbols símbolos.
    """
    symbols = sorted(probs.keys())
    cum_probs = {}
    cum = 0.0
    for s in symbols:
        cum_probs[s] = (cum, cum + probs[s])
        cum += probs[s]
    
    decoded = []
    low, high = 0.0, 1.0
    
    for _ in range(n_symbols):
        width = high - low
        # Encontrar el símbolo cuyo subintervalo contiene code_value
        for s in symbols:
            s_low, s_high = cum_probs[s]
            new_low = low + width * s_low
            new_high = low + width * s_high
            if new_low <= code_value < new_high:
                decoded.append(s)
                low, high = new_low, new_high
                break
    
    return decoded

## 2. Ejemplo del artículo: fuente binaria P(0)=0.8, P(1)=0.2

In [ ]:
probs = {'0': 0.8, '1': 0.2}
message = list('011')

low, high, bits, steps = arithmetic_encode(message, probs)

print(f"Mensaje: {''.join(message)}")
print(f"Probabilidad del mensaje: P(011) = 0.8 × 0.2 × 0.2 = {0.8*0.2*0.2:.4f}")
print(f"Intervalo final: [{low:.6f}, {high:.6f})")
print(f"Longitud del intervalo: {high-low:.6f}")
print(f"Bits necesarios: {bits}")
print(f"Entropía teórica de '011': {-math.log2(0.8*0.2*0.2):.3f} bits")

print("\nPasos de la codificación:")
for i, step in enumerate(steps):
    before = step['interval_before']
    after = step['interval_after']
    print(f"  Símbolo '{step['symbol']}': [{before[0]:.4f}, {before[1]:.4f}) → [{after[0]:.4f}, {after[1]:.4f})")

In [ ]:
# Verificar decodificación
# Usamos el punto medio del intervalo como código
code_value = (low + high) / 2
decoded = arithmetic_decode(code_value, probs, len(message))
print(f"Código transmitido: {code_value:.8f}")
print(f"Mensaje decodificado: {''.join(decoded)}")
print(f"¿Correcto? {decoded == message}")

## 3. Comparación con Huffman

In [ ]:
def huffman_codes(probs):
    """Calcula los códigos de Huffman para un diccionario de probabilidades."""
    # Min-heap: (probabilidad, símbolo_o_árbol)
    heap = [(p, [s]) for s, p in probs.items()]
    heapq.heapify(heap)
    codes = {s: '' for s in probs}
    
    if len(heap) == 1:
        return {list(probs.keys())[0]: '0'}
    
    # Construir árbol
    trees = {s: s for s in probs}  # árbol trivial
    
    # Implementación simplificada: solo calcula longitudes óptimas
    # usando la construcción por prioridades
    items = sorted(probs.items(), key=lambda x: x[1])
    
    def build(symbols_probs):
        if len(symbols_probs) == 1:
            return {symbols_probs[0][0]: ''}
        if len(symbols_probs) == 2:
            return {symbols_probs[0][0]: '0', symbols_probs[1][0]: '1'}
        # Combinar los dos de menor probabilidad
        sorted_sp = sorted(symbols_probs, key=lambda x: x[1])
        s1, p1 = sorted_sp[0]
        s2, p2 = sorted_sp[1]
        rest = sorted_sp[2:] + [((s1, s2), p1 + p2)]
        sub = build(rest)
        result = {}
        for k, v in sub.items():
            if k == (s1, s2):
                result[s1] = v + '0'
                result[s2] = v + '1'
            else:
                result[k] = v
        return result
    
    return build(list(probs.items()))


def compare_arithmetic_vs_huffman(probs, messages):
    """Compara la eficiencia de codificación aritmética vs Huffman."""
    entropy = -sum(p * math.log2(p) for p in probs.values())
    huff = huffman_codes(probs)
    huff_lengths = {s: len(c) for s, c in huff.items()}
    
    print(f"Entropía de la fuente: {entropy:.4f} bits/símbolo")
    print(f"Códigos de Huffman: {huff}")
    print(f"Longitud media Huffman: {sum(probs[s]*huff_lengths[s] for s in probs):.4f} bits/símbolo")
    print()
    
    print(f"{'Mensaje':<15} {'n':<5} {'Teoría':<10} {'Huffman':<10} {'Aritmético':<12} {'Ratio Huff':<12} {'Ratio Arit'}")
    print('-' * 80)
    
    for msg in messages:
        n = len(msg)
        theory = -math.log2(max(1e-10, math.prod(probs[s] for s in msg)))
        huff_bits = sum(huff_lengths[s] for s in msg)
        _, _, arit_bits, _ = arithmetic_encode(msg, probs)
        
        ratio_h = huff_bits / theory if theory > 0 else float('inf')
        ratio_a = arit_bits / theory if theory > 0 else float('inf')
        
        print(f"{''.join(msg):<15} {n:<5} {theory:<10.3f} {huff_bits:<10} {arit_bits:<12} {ratio_h:<12.3f} {ratio_a:.3f}")


probs = {'0': 0.8, '1': 0.2}
messages = [
    list('0'),
    list('1'),
    list('00'),
    list('01'),
    list('011'),
    list('000'),
    list('0000'),
    list('00000000'),
    list('11111111'),
]

compare_arithmetic_vs_huffman(probs, messages)

## 4. Eficiencia vs longitud del mensaje

In [ ]:
import random

def measure_efficiency(probs, n_lengths, n_samples=100):
    """Mide la eficiencia media de codificación aritmética vs Huffman para mensajes de distinta longitud."""
    entropy = -sum(p * math.log2(p) for p in probs.values())
    huff = huffman_codes(probs)
    huff_lengths = {s: len(c) for s, c in huff.items()}
    
    symbols = list(probs.keys())
    ps = [probs[s] for s in symbols]
    
    results = []
    for n in n_lengths:
        arit_rates, huff_rates = [], []
        for _ in range(n_samples):
            msg = random.choices(symbols, weights=ps, k=n)
            _, _, arit_bits, _ = arithmetic_encode(msg, probs)
            huff_bits = sum(huff_lengths[s] for s in msg)
            arit_rates.append(arit_bits / n)
            huff_rates.append(huff_bits / n)
        results.append({
            'n': n,
            'arit_mean': np.mean(arit_rates),
            'huff_mean': np.mean(huff_rates),
            'entropy': entropy
        })
    return results


random.seed(42)
probs_skewed = {'a': 0.5, 'b': 0.25, 'c': 0.125, 'd': 0.125}
entropy_skewed = -sum(p * math.log2(p) for p in probs_skewed.values())

lengths = [1, 2, 5, 10, 20, 50, 100]
results = measure_efficiency(probs_skewed, lengths)

print(f"Fuente: P(a)=0.5, P(b)=0.25, P(c)=P(d)=0.125")
print(f"Entropía: {entropy_skewed:.4f} bits/símbolo")
print()
print(f"{'n':<8} {'Aritmético':<14} {'Huffman':<14} {'Exceso Arit':<14} {'Exceso Huff'}")
print('-' * 65)
for r in results:
    exc_a = r['arit_mean'] - r['entropy']
    exc_h = r['huff_mean'] - r['entropy']
    print(f"{r['n']:<8} {r['arit_mean']:<14.4f} {r['huff_mean']:<14.4f} {exc_a:<14.4f} {exc_h:.4f}")
print()
print("Conclusión: el exceso de aritmético tiende a 0 más rápido (≈ 2/n).")
print("Huffman tiene exceso ≤ 1 bit/símbolo pero no converge tan rápido para n grande.")

## 5. Codificación adaptativa

Las probabilidades se actualizan conforme se va codificando el mensaje.

In [ ]:
def adaptive_arithmetic_encode(message, alphabet):
    """
    Codificación aritmética adaptativa: actualiza las probabilidades con cada símbolo.
    
    Usa estimación de frecuencia con suavizado de Laplace (conteo inicial = 1).
    """
    counts = {s: 1 for s in alphabet}  # Laplace smoothing
    total = len(alphabet)
    
    low, high = 0.0, 1.0
    total_bits = 0
    
    for i, symbol in enumerate(message):
        # Probabilidades actuales
        probs = {s: counts[s] / total for s in alphabet}
        
        # Codificar símbolo
        symbols = sorted(alphabet)
        width = high - low
        cum = 0.0
        for s in symbols:
            s_width = width * probs[s]
            if s == symbol:
                high = low + cum + s_width
                low = low + cum
                break
            cum += s_width
        
        # Actualizar modelo
        counts[symbol] += 1
        total += 1
    
    prob_message = high - low
    bits = math.ceil(-math.log2(max(prob_message, 1e-300))) + 1
    return bits


# Texto de prueba: frecuencias del español aproximadas
text_sample = 'aaabbbcccdddeeefffggg'
alphabet = sorted(set(text_sample))

# Probabilidades reales del texto
counts = Counter(text_sample)
n = len(text_sample)
true_probs = {c: counts[c]/n for c in alphabet}
entropy_bits = -sum(p * math.log2(p) for p in true_probs.values()) * n

adaptive_bits = adaptive_arithmetic_encode(list(text_sample), alphabet)

print(f"Texto: '{text_sample}'")
print(f"Longitud: {n} símbolos")
print(f"Entropía teórica total: {entropy_bits:.2f} bits")
print(f"Codificación adaptativa: {adaptive_bits} bits")
print(f"Overhead adaptativo: {adaptive_bits - entropy_bits:.2f} bits ({(adaptive_bits/entropy_bits - 1)*100:.1f}% extra)")
print()
print("El overhead adaptativo disminuye a medida que el modelo aprende la distribución.")